# B8 - Simple Transformer for Next-Word Prediction
- Max length 10, embedding 32, positional embedding 32
- Self-attention with 2 heads x 16 dims, written by hand with Q, K, V and a **causal mask**
- FFN 64 ReLU, 1 block, output Dense + Softmax
- Cross-entropy, Adam, 10 epochs, attention heatmap

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
tf.keras.utils.set_random_seed(0)

sentences = [
    "i love deep learning", "i love machine learning", "deep learning is fun",
    "machine learning is powerful", "i like to learn new things", "students love to learn",
    "the cat sat on the mat", "the dog sat on the rug", "the cat is very cute",
    "the dog is very loyal", "i like the cat", "i like the dog",
    "transformers use self attention", "attention is all you need", "self attention looks at every word",
    "we train the model with adam", "the model predicts the next word", "deep models need data",
    "data is very important", "learning new things is fun",
]

# Tokenize
words = sorted(set(" ".join(sentences).split()))
word2idx = {w: i + 1 for i, w in enumerate(words)}       # 0 = <pad>
word2idx["<pad>"] = 0
idx2word = {i: w for w, i in word2idx.items()}
VOCAB = len(word2idx)
MAX_LEN = 10

def encode(s):
    return [word2idx[w] for w in s.split()]

# Input = words[:-1], target = words[1:]  (predict next word at every position)
X, Y = [], []
for s in sentences:
    ids = encode(s)
    X.append(ids[:-1]); Y.append(ids[1:])
X = keras.preprocessing.sequence.pad_sequences(X, maxlen=MAX_LEN, padding='post')
Y = keras.preprocessing.sequence.pad_sequences(Y, maxlen=MAX_LEN, padding='post')
print("Vocab size:", VOCAB, "| X shape:", X.shape)

In [ ]:
class CausalSelfAttention(layers.Layer):
    def __init__(self, d_model=32, num_heads=2):
        super().__init__()
        self.h, self.d_model = num_heads, d_model
        self.depth = d_model // num_heads          # 16 per head
        self.wq = layers.Dense(d_model)
        self.wk = layers.Dense(d_model)
        self.wv = layers.Dense(d_model)
        self.wo = layers.Dense(d_model)

    def split_heads(self, x):
        b = tf.shape(x)[0]
        x = tf.reshape(x, (b, -1, self.h, self.depth))
        return tf.transpose(x, [0, 2, 1, 3])        # (B, heads, L, depth)

    def call(self, x):
        b, L = tf.shape(x)[0], tf.shape(x)[1]
        q = self.split_heads(self.wq(x))
        k = self.split_heads(self.wk(x))
        v = self.split_heads(self.wv(x))
        # Scaled dot-product attention
        scores = tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(float(self.depth))   # (B, h, L, L)
        causal = tf.linalg.band_part(tf.ones((L, L)), -1, 0)                         # lower triangle = 1
        scores = scores + (1.0 - causal) * -1e9                                       # block future words
        weights = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(weights, v)                                                   # (B, h, L, depth)
        out = tf.reshape(tf.transpose(out, [0, 2, 1, 3]), (b, -1, self.d_model))
        return self.wo(out), weights


class MiniTransformer(keras.Model):
    def __init__(self, vocab, max_len=10, d_model=32, heads=2, ff_dim=64):
        super().__init__()
        self.tok_emb = layers.Embedding(vocab, d_model)
        self.pos_emb = layers.Embedding(max_len, d_model)
        self.attn = CausalSelfAttention(d_model, heads)
        self.norm1 = layers.LayerNormalization()
        self.ff1 = layers.Dense(ff_dim, activation='relu')
        self.ff2 = layers.Dense(d_model)
        self.norm2 = layers.LayerNormalization()
        self.out = layers.Dense(vocab, activation='softmax')

    def call(self, x):
        L = tf.shape(x)[1]
        h = self.tok_emb(x) + self.pos_emb(tf.range(L))
        a, weights = self.attn(h)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ff2(self.ff1(h)))
        return self.out(h), weights

model = MiniTransformer(VOCAB, MAX_LEN)

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction='none')
optimizer = keras.optimizers.Adam(0.01)
dataset = tf.data.Dataset.from_tensor_slices((X, Y)).shuffle(100).batch(4)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        probs, _ = model(x)
        mask = tf.cast(y != 0, tf.float32)            # ignore padding
        loss = tf.reduce_sum(loss_fn(y, probs) * mask) / tf.reduce_sum(mask)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

losses = []
for epoch in range(10):
    ep = np.mean([train_step(x, y) for x, y in dataset])
    losses.append(ep)
    print(f"Epoch {epoch+1}: loss = {ep:.4f}")
plt.plot(losses); plt.xlabel('epoch'); plt.ylabel('cross-entropy'); plt.show()

In [ ]:
def predict_next_word(sentence):
    ids = encode(sentence)
    x = keras.preprocessing.sequence.pad_sequences([ids], maxlen=MAX_LEN, padding='post')
    probs, weights = model(x)
    next_id = int(tf.argmax(probs[0, len(ids) - 1]))
    return idx2word[next_id], weights

for s in ["i love deep", "the cat sat on the", "attention is all you", "the model predicts the next"]:
    w, _ = predict_next_word(s)
    print(f"'{s}' -> '{w}'")

In [ ]:
# Attention heatmap
sentence = "the cat sat on the"
_, weights = predict_next_word(sentence)
tokens = sentence.split(); n = len(tokens)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for head in range(2):
    w = weights[0, head, :n, :n].numpy()
    im = axes[head].imshow(w, cmap='viridis')
    axes[head].set_xticks(range(n)); axes[head].set_xticklabels(tokens, rotation=45)
    axes[head].set_yticks(range(n)); axes[head].set_yticklabels(tokens)
    axes[head].set_title(f'Head {head+1}'); axes[head].set_xlabel('Key'); axes[head].set_ylabel('Query')
    fig.colorbar(im, ax=axes[head])
plt.tight_layout(); plt.show()
# The upper triangle is 0 because of the causal mask (no attention to future words)